# Confluent Cloud Lightning Query: Customer 360 Profile (`fct_c360_profile`)

This notebook demonstrates how to connect to **Confluent Cloud** and query the **`fct_c360_profile`** Lightning Table via the **Real-Time Context Engine (RTCE) / Lightning Query API**.

### Background & Architecture
- **Lightning Tables**: Managed materialization of a Kafka topic, optimized for sub-second, point-lookup queries.
- **`fct_c360_profile`**: Customer 360 fact table maintained by Flink SQL (`pipelines/models/c360/facts/fct_c360_profile.sql`), joining `dim_customers` and `dim_tx_x_account` into a consolidated, upserted profile per customer.
- **Endpoint**: `https://sql.{CLOUD_REGION}.{CLOUD_PROVIDER}.confluent.cloud/query/v1alpha1`
- **Authentication**: HTTP Basic Auth using an Organization-wide **Global API Key** (`CONFLUENT_GLOBAL_KEY` / `CONFLUENT_GLOBAL_SECRET`), provisioned by `IaC/rtce`.
- **Scope**: `catalog_name` = Environment ID (`FLINK_ENV_ID`), `database_name` = Kafka Cluster ID (`KAFKA_CLUSTER_ID`).

## 1. Import Dependencies

Import required libraries (`httpx`, `pandas`, `dotenv`, etc.).

In [1]:
import os
import json
import time
from pathlib import Path
from typing import Sequence, Any
import httpx
import pandas as pd
from dotenv import load_dotenv

print(f"httpx version: {httpx.__version__}")
print(f"pandas version: {pd.__version__}")

httpx version: 0.28.1
pandas version: 3.1.1


## 2. Load Configuration & Environment Variables

The credentials can be loaded automatically from `apps/backend/.env` (or project root `.env`), or set directly via environment variables.

In [2]:
# Search for .env in apps/backend or parent directories
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
backend_env = repo_root / "apps" / "backend" / ".env"
root_env = repo_root / ".env"

if backend_env.exists():
    load_dotenv(dotenv_path=backend_env, override=False)
    print(f"Loaded environment from: {backend_env}")
elif root_env.exists():
    load_dotenv(dotenv_path=root_env, override=False)
    print(f"Loaded environment from: {root_env}")
else:
    print("No .env file found; using active environment variables.")

# Extract Confluent Lightning / RTCE configuration
CLOUD_REGION = os.getenv("CLOUD_REGION", "us-west-2")
CLOUD_PROVIDER = os.getenv("CLOUD_PROVIDER", "aws")
FLINK_ENV_ID = os.getenv("FLINK_ENV_ID", "")
KAFKA_CLUSTER_ID = os.getenv("KAFKA_CLUSTER_ID", "")
CONFLUENT_GLOBAL_KEY = os.getenv("CONFLUENT_GLOBAL_KEY", "")
CONFLUENT_GLOBAL_SECRET = os.getenv("CONFLUENT_GLOBAL_SECRET", "")

def mask_key(k: str) -> str:
    return f"{k[:6]}...{k[-4:]}" if len(k) > 10 else ("<unset>" if not k else "***")

print("\n--- Confluent Lightning Configuration ---")
print(f"Cloud Provider  : {CLOUD_PROVIDER}")
print(f"Cloud Region    : {CLOUD_REGION}")
print(f"Environment ID  : {FLINK_ENV_ID or '<unset>'}")
print(f"Kafka Cluster ID: {KAFKA_CLUSTER_ID or '<unset>'}")
print(f"Global Key      : {mask_key(CONFLUENT_GLOBAL_KEY)}")
print(f"Global Secret   : {mask_key(CONFLUENT_GLOBAL_SECRET)}")

missing = [k for k, v in [
    ("CLOUD_REGION", CLOUD_REGION),
    ("CLOUD_PROVIDER", CLOUD_PROVIDER),
    ("FLINK_ENV_ID", FLINK_ENV_ID),
    ("KAFKA_CLUSTER_ID", KAFKA_CLUSTER_ID),
    ("CONFLUENT_GLOBAL_KEY", CONFLUENT_GLOBAL_KEY),
    ("CONFLUENT_GLOBAL_SECRET", CONFLUENT_GLOBAL_SECRET)
] if not v]

if missing:
    print(f"\n⚠️ Warning: Missing required variables: {', '.join(missing)}")
    print("Run `./scripts/set_env_from_tf.sh` or configure them in `apps/backend/.env`.")
else:
    print("\nAll Lightning credentials are set.")

Loaded environment from: /Users/jerome/Documents/Code/streamhouse_c360_demo/apps/backend/.env

--- Confluent Lightning Configuration ---
Cloud Provider  : aws
Cloud Region    : us-west-2
Environment ID  : env-w0gk9w
Kafka Cluster ID: lkc-j5r8858
Global Key      : IOGMHH...7GPE
Global Secret   : cflt7T...RIaQ

All Lightning credentials are set.


## 3. Confluent Lightning Client Implementation

Define a helper function to send SQL queries to Confluent Cloud's Lightning Query API and return the results as a `pandas.DataFrame`.

In [3]:
def query_lightning(
    sql: str,
    max_rows: int = 100,
    timeout_seconds: float = 10.0
) -> pd.DataFrame:
    """Execute a SQL query against Confluent Cloud Lightning Query API.
    
    Args:
        sql: SQL query string (e.g. 'SELECT * FROM fct_c360_profile LIMIT 10')
        max_rows: Maximum result rows to return
        timeout_seconds: HTTP request timeout
        
    Returns:
        pandas.DataFrame containing the query results.
    """
    endpoint = f"https://sql.{CLOUD_REGION}.{CLOUD_PROVIDER}.confluent.cloud/query/v1alpha1"
    
    payload = {
        "catalog_name": FLINK_ENV_ID,
        "database_name": KAFKA_CLUSTER_ID,
        "query": sql,
        "options": {
            "max_result_rows": max_rows,
            "result_format": "JSON"
        },
        "client_info": {
            "client_name": "c360_jupyter_notebook"
        }
    }
    
    start_time = time.perf_counter()
    
    try:
        response = httpx.post(
            endpoint,
            json=payload,
            auth=(CONFLUENT_GLOBAL_KEY, CONFLUENT_GLOBAL_SECRET),
            headers={"Content-Type": "application/json"},
            timeout=timeout_seconds
        )
        duration_ms = (time.perf_counter() - start_time) * 1000
        
        if response.status_code in (401, 403):
            raise PermissionError(
                f"Authentication error ({response.status_code}): Ensure CONFLUENT_GLOBAL_KEY is a "
                "GLOBAL Organization-scoped API key created by IaC/rtce, not a cluster or Flink key.\n"
                f"Response: {response.text}"
            )
        response.raise_for_status()
        
        data_json = response.json()
        result = data_json.get("result", {})
        schema_info = result.get("schema", {})
        data_rows = result.get("data", [])
        
        # Extract column names and normalize to lowercase snake_case
        columns = [
            (c if isinstance(c, str) else c.get("name", f"col_{idx}")).lower()
            for idx, c in enumerate(schema_info.get("columns", []))
        ]
        
        df = pd.DataFrame(data_rows, columns=columns if columns else None)
        print(f" Query succeeded in {duration_ms:.2f} ms ({len(df)} rows returned)")
        return df

    except httpx.HTTPStatusError as err:
        print(f"❌ HTTP {err.response.status_code} Error: {err.response.text}")
        raise
    except Exception as err:
        print(f"❌ Error executing Lightning query: {err}")
        raise

## 4. Query Sample Profiles from `fct_c360_profile`

Let's execute a `SELECT *` query on the `fct_c360_profile` Lightning Table to inspect available customer attributes.

In [4]:
sample_sql = "SELECT * FROM fct_c360_profile LIMIT 10"
print(f"Executing: {sample_sql}")

df_sample = query_lightning(sample_sql, max_rows=10)
df_sample.head()

Executing: SELECT * FROM fct_c360_profile LIMIT 10
 Query succeeded in 559.60 ms (10 rows returned)


,customer_id,first_name,last_name,date_of_birth,birth_year,birth_month,customer_segment,customer_since,gender,city,...,income_band,missing_email_flag,missing_phone_flag,card_brand,card_type,card_credit_limit,account_count,total_balance,txn_count,txn_total
0,00000000-0000-0000-0000-234567890124,David,Okafor,1982-09-30,1982,9,SMB,2019-07-04 00:00:00+00,M,Chicago,...,LOW,0,0,None,None,None,1,35943,2,464
1,00000000-0000-0000-0000-567890123457,Ashley,Nguyen,1995-12-05,1995,12,PRIVATE,2023-08-09 00:00:00+00,F,Ann Arbor,...,MIDDLE,0,0,None,None,None,1,1482,2,454
2,00000000-0000-0000-0000-456789012346,Michael,Torres,1975-06-22,1975,6,PRIVATE,2018-04-17 00:00:00+00,M,Miami,...,HIGH,0,0,None,None,None,1,88361,3,2695
3,00000000-0000-0000-0000-012345678902,Robert,Kim,1978-11-03,1978,11,PRIVATE,2020-09-10 00:00:00+00,M,New York,...,HIGH,0,0,None,None,None,1,8898,6,2592
4,00000000-0000-0000-0000-f01234567891,Maria,Gonzalez,1990-07-24,1990,7,SMB,2021-03-15 00:00:00+00,F,Los Angeles,...,MIDDLE,0,0,None,None,None,1,36955,4,608


## 5. Sub-Second Point Lookup by Customer ID

The primary strength of Lightning Tables is ultra-low latency point lookups on primary keys (`customer_id`).
Here we take a customer from the sample query and execute a point lookup.

In [5]:
target_customer_id = df_sample.iloc[0]["customer_id"] if not df_sample.empty else "CUST-1001"

point_lookup_sql = f"""
SELECT 
    customer_id, 
    first_name, 
    last_name, 
    customer_segment, 
    generation_segment, 
    yearly_income, 
    credit_score, 
    card_brand, 
    account_count, 
    total_balance, 
    txn_count, 
    txn_total
FROM fct_c360_profile 
WHERE customer_id = '{target_customer_id}'
""".strip()

print(f"Executing Point Lookup:\n{point_lookup_sql}\n")
df_profile = query_lightning(point_lookup_sql, max_rows=1)
df_profile.T

Executing Point Lookup:
SELECT 
    customer_id, 
    first_name, 
    last_name, 
    customer_segment, 
    generation_segment, 
    yearly_income, 
    credit_score, 
    card_brand, 
    account_count, 
    total_balance, 
    txn_count, 
    txn_total
FROM fct_c360_profile 
WHERE customer_id = '00000000-0000-0000-0000-234567890124'

 Query succeeded in 280.48 ms (1 rows returned)


,0
customer_id,00000000-0000-0000-0000-234567890124
first_name,David
last_name,Okafor
customer_segment,SMB
generation_segment,Gen X
yearly_income,64300.00
credit_score,710
card_brand,None
account_count,1
total_balance,35943


## 6. Segment Analysis Query

Query customer profiles across customer segments (e.g., `VIP`, `Premium`, `Standard`) to analyze credit scores, balances, and transaction totals.

In [6]:
segment_sql = """
SELECT 
    customer_id,
    first_name,
    last_name,
    customer_segment,
    generation_segment,
    city,
    state,
    yearly_income,
    credit_score,
    total_balance,
    txn_count,
    txn_total
FROM fct_c360_profile
LIMIT 50
""".strip()

df_customers = query_lightning(segment_sql, max_rows=50)

# Convert numeric columns if necessary
numeric_cols = ["yearly_income", "credit_score", "total_balance", "txn_count", "txn_total"]
for col in numeric_cols:
    if col in df_customers.columns:
        df_customers[col] = pd.to_numeric(df_customers[col], errors="coerce")

print("\nSummary statistics by Customer Segment:")
if "customer_segment" in df_customers.columns:
    display_summary = df_customers.groupby("customer_segment")[["credit_score", "total_balance", "txn_total"]].mean()
    print(display_summary)

df_customers.head(10)

 Query succeeded in 253.25 ms (10 rows returned)

Summary statistics by Customer Segment:
                  credit_score  total_balance  txn_total
customer_segment                                        
PRIVATE                 715.00        43433.5     1658.5
SMB                     678.25        20767.5      523.5


,customer_id,first_name,last_name,customer_segment,generation_segment,city,state,yearly_income,credit_score,total_balance,txn_count,txn_total
0,00000000-0000-0000-0000-678901234568,Thomas,Brennan,SMB,Boomer+,Seattle,WA,59800.50,620,0,0,0
1,00000000-0000-0000-0000-234567890124,David,Okafor,SMB,Gen X,Chicago,IL,64300.00,710,35943,2,464
2,00000000-0000-0000-0000-567890123457,Ashley,Nguyen,PRIVATE,Millennial,Ann Arbor,MI,102400.00,730,1482,2,454
3,00000000-0000-0000-0000-456789012346,Michael,Torres,PRIVATE,Gen X,Miami,FL,155000.00,772,88361,3,2695
4,00000000-0000-0000-0000-012345678902,Robert,Kim,PRIVATE,Gen X,New York,NY,142300.00,801,8898,6,2592
5,00000000-0000-0000-0000-f01234567891,Maria,Gonzalez,SMB,Millennial,Los Angeles,CA,96200.50,688,36955,4,608
6,00000000-0000-0000-0000-345678901235,Jennifer,Walsh,PRIVATE,Millennial,Boston,MA,87900.75,590,149370,4,1104
7,00000000-0000-0000-0000-789012345679,Linda,Sharma,SMB,Millennial,Austin,TX,71250.00,695,10172,6,1022
8,00000000-0000-0000-0000-123456789013,Susan,Patel,PRIVATE,Millennial,San Francisco,CA,118750.25,655,12017,2,2065
9,00000000-0000-0000-0000-ef1234567890,James,Carter,PRIVATE,Gen X,Springfield,IL,78500.00,742,473,5,1041


## 7. Profile Attributes Overview

Inspect the schema and columns returned from `fct_c360_profile`.

In [7]:
if not df_customers.empty:
    print(f"Total Columns ({len(df_customers.columns)}): {list(df_customers.columns)}")
    print("\nData Types:")
    print(df_customers.dtypes)
else:
    print("No data returned to inspect.")

Total Columns (12): ['customer_id', 'first_name', 'last_name', 'customer_segment', 'generation_segment', 'city', 'state', 'yearly_income', 'credit_score', 'total_balance', 'txn_count', 'txn_total']

Data Types:
customer_id               str
first_name                str
last_name                 str
customer_segment          str
generation_segment        str
city                      str
state                     str
yearly_income         float64
credit_score            int64
total_balance           int64
txn_count               int64
txn_total               int64
dtype: object
